# Publication copy

Outputs and machine-specific paths were removed. Run `python scripts/prepare_local_artifacts.py` first. GPU experiments additionally require datasets, authorized pretrained models and the large artifacts listed in `docs/LARGE_ARTIFACTS.md`. Do not overwrite the frozen reporting inputs.


In [ ]:
from pathlib import Path
import os, sys
_start = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).expanduser().resolve()
_PUBLICATION_ROOT = next((p for p in (_start, *_start.parents)
                         if (p / "Methods").is_dir() and (p / "README.md").is_file()), None)
if _PUBLICATION_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT to the cloned cancer_image_pathology folder")
os.chdir(_PUBLICATION_ROOT)
sys.path.insert(0, str(_PUBLICATION_ROOT))
os.environ["PROJECT_ROOT"] = str(_PUBLICATION_ROOT)
print("Project:", _PUBLICATION_ROOT)


# Final three-model classification and attribution comparison

**Primary comparison:** ResNet18 + Grad-CAM versus DINOv2 + gradient-weighted rollout versus UNI + gradient-weighted rollout.

This notebook is a results-only stage. It reuses the grouped OOF predictions, the frozen 272-image faithfulness cohort, deletion and input-occlusion results, and ten-seed stability outputs. It does not train or alter any model.

The completed UNI activation-patching experiment is retained as a secondary negative-result ablation. Its 55-error analysis is explicitly exploratory. Only penultimate-layer, within-image-mean activation patching was tested, and no causal-ranking loss is implemented.

Highlighted patches are regions contributing to a model prediction under a particular explanation method. They are not biological causes or validated pathology annotations.


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)


In [ ]:
def locate_project_root():
    return _PUBLICATION_ROOT


PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'final_three_model_comparison'
TABLE_DIR = OUTPUT_DIR / 'tables'
FIGURE_DIR = OUTPUT_DIR / 'figures'
for directory in (TABLE_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print('Project:', PROJECT_ROOT)
print('Final outputs:', OUTPUT_DIR)


In [ ]:
from Methods.FinalComparison import (
    FAITHFULNESS_METRICS,
    FIXED_DELETION_METRICS,
    classification_seed_tests,
    classification_source_tests,
    classification_summary,
    create_final_figures,
    faithfulness_summary,
    focused_faithfulness_tests,
    incorrect_intervention_exploratory,
    load_intervention_artifacts,
    load_three_model_artifacts,
    primary_faithfulness_rows,
    stability_pairwise_tests,
    stability_summary,
)

BOOTSTRAP_ITERATIONS = 5000
RANDOM_SEED = 2027
REQUIRE_INTERVENTION_ARTIFACTS = True


## 1. Load and validate frozen artifacts

All three primary models must contain the same 272 cohort IDs, eight classes, and ten source cases. The intervention section reads the per-image CSVs written by notebook 06; the executed notebook alone contains only aggregate displays and is insufficient for paired resampling.


In [ ]:
artifacts = load_three_model_artifacts(PROJECT_ROOT)
try:
    intervention = load_intervention_artifacts(PROJECT_ROOT)
except FileNotFoundError:
    if REQUIRE_INTERVENTION_ARTIFACTS:
        raise
    intervention = None

cohort = artifacts['cohort']
print('Frozen cohort:', len(cohort), 'images,', cohort['class_name'].nunique(),
      'classes,', cohort['case_id'].nunique(), 'source cases')
print('Classification seeds:', sorted(
    artifacts['classification'].query("scope == 'aggregate_oof'")['seed'].unique()
))
if intervention is not None:
    assert set(intervention['metrics']['cohort_id']) == set(cohort['cohort_id'])
    print('Intervention per-image artifacts: available')


## 2. Classification performance


In [ ]:
classification_table = classification_summary(artifacts['classification'])
classification_tests, classification_pairs = classification_seed_tests(
    artifacts['classification'],
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
    random_seed=RANDOM_SEED,
)
classification_source_test_table = classification_source_tests(
    artifacts['per_source'],
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
    random_seed=RANDOM_SEED,
)
per_class_classification = (
    artifacts['per_class']
    .groupby(['model', 'label', 'class_name'])
    .agg(
        seeds=('seed', 'nunique'),
        images_per_seed=('image_count', 'first'),
        accuracy_mean=('accuracy', 'mean'),
        accuracy_std=('accuracy', 'std'),
    )
    .reset_index()
)
classification_table.to_csv(TABLE_DIR / '01_classification_summary.csv', index=False)
classification_tests.to_csv(TABLE_DIR / '02_classification_seed_paired_tests.csv', index=False)
classification_source_test_table.to_csv(
    TABLE_DIR / '03_classification_source_hierarchical_tests.csv', index=False
)
per_class_classification.to_csv(TABLE_DIR / '04_per_class_classification.csv', index=False)
display(classification_table.style.format(precision=4))
display(classification_tests.style.format(precision=4))


## 3. Primary faithfulness comparison

The primary target is the true class for every image. This avoids pairing explanations for different predicted classes. Lower top-minus-random AUC is better because deleting highly ranked patches should reduce the target score more rapidly than random deletion.


In [ ]:
primary = primary_faithfulness_rows(artifacts['faithfulness'])
faithfulness_table = faithfulness_summary(primary)
faithfulness_tests = focused_faithfulness_tests(
    artifacts['faithfulness_tests']
)
per_class_faithfulness = (
    primary.groupby(['model', 'true_class', 'class_name'])
    [[*FAITHFULNESS_METRICS, *FIXED_DELETION_METRICS]]
    .agg(['count', 'mean', 'std'])
)
per_class_faithfulness.columns = [
    f'{metric}_{stat}' for metric, stat in per_class_faithfulness.columns
]
per_class_faithfulness = per_class_faithfulness.reset_index()
faithfulness_table.to_csv(TABLE_DIR / '05_primary_faithfulness_summary.csv', index=False)
faithfulness_tests.to_csv(TABLE_DIR / '06_primary_faithfulness_paired_tests.csv', index=False)
per_class_faithfulness.to_csv(TABLE_DIR / '07_per_class_faithfulness.csv', index=False)
artifacts['within_model_tests'].to_csv(
    TABLE_DIR / '08_within_model_top_vs_random_tests.csv', index=False
)
display(faithfulness_table.style.format(precision=4))
display(faithfulness_tests.style.format(precision=4))


## 4. Exploratory analysis of 55 incorrect UNI predictions

This is a secondary, explicitly exploratory family. It compares intervention attribution with UNI gradient-weighted rollout for the true class on exactly the 55 images UNI misclassified at the reference seed. Paired Wilcoxon tests and Holm adjustment are reported alongside source-case hierarchical bootstrap intervals.


In [ ]:
if intervention is not None:
    incorrect_tests, incorrect_pairs, incorrect_by_class = (
        incorrect_intervention_exploratory(
            intervention['metrics'],
            bootstrap_iterations=BOOTSTRAP_ITERATIONS,
            random_seed=RANDOM_SEED,
            expected_images=55,
        )
    )
    incorrect_tests.to_csv(
        TABLE_DIR / '09_exploratory_incorrect_uni_intervention_tests.csv', index=False
    )
    incorrect_pairs.to_csv(
        TABLE_DIR / '10_exploratory_incorrect_uni_paired_values.csv', index=False
    )
    incorrect_by_class.to_csv(
        TABLE_DIR / '11_exploratory_incorrect_uni_by_class.csv', index=False
    )
    display(incorrect_tests.style.format(precision=4))
else:
    incorrect_tests = None
    print('Intervention artifacts unavailable; exploratory analysis not run.')


## 5. Cross-seed prediction and attribution stability


In [ ]:
stability_table = stability_summary(artifacts['stability'])
stability_tests = stability_pairwise_tests(
    artifacts['stability'],
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
    random_seed=RANDOM_SEED,
)
per_class_stability = (
    artifacts['stability']
    .groupby(['model', 'class_name'])[[
        'pairwise_prediction_agreement',
        'mean_predicted_class_spearman_same_prediction',
        'mean_common_true_class_spearman',
    ]]
    .agg(['count', 'mean', 'std'])
)
per_class_stability.columns = [
    f'{metric}_{stat}' for metric, stat in per_class_stability.columns
]
per_class_stability = per_class_stability.reset_index()
stability_table.to_csv(TABLE_DIR / '12_stability_summary.csv', index=False)
stability_tests.to_csv(TABLE_DIR / '13_stability_paired_tests.csv', index=False)
per_class_stability.to_csv(TABLE_DIR / '14_per_class_stability.csv', index=False)
display(stability_table.style.format(precision=4))
display(stability_tests.style.format(precision=4))


## 6. Secondary intervention ablation

This section is not part of the primary three-model comparison. It reports only the tested penultimate-layer, within-image-mean activation intervention. Failure of this ablation does not establish that every activation-patching design is inferior.


In [ ]:
if intervention is not None:
    intervention_true = intervention['metrics'][
        intervention['metrics']['target_class'].eq(
            intervention['metrics']['true_class']
        )
    ]
    intervention_ablation_summary = (
        intervention_true.groupby('method')[[
            'attribution_occlusion_spearman',
            'top_minus_random_target_logit_auc',
            'top_minus_random_margin_auc',
            'top_5_target_logit_drop',
            'top_10_target_logit_drop',
            'top_20_target_logit_drop',
            'top_30_target_logit_drop',
            'top_5_margin_drop',
            'top_10_margin_drop',
            'top_20_margin_drop',
            'top_30_margin_drop',
        ]]
        .agg(['count', 'mean', 'std'])
    )
    intervention_ablation_summary.columns = [
        f'{metric}_{stat}' for metric, stat in intervention_ablation_summary.columns
    ]
    intervention_ablation_summary = intervention_ablation_summary.reset_index()
    intervention_ablation_summary.to_csv(
        TABLE_DIR / '15_intervention_negative_ablation_summary.csv', index=False
    )
    intervention['stability'].groupby('method')[[
        'pairwise_prediction_agreement',
        'mean_predicted_class_spearman_same_prediction',
        'mean_common_true_class_spearman',
    ]].mean().reset_index().to_csv(
        TABLE_DIR / '16_intervention_stability_summary.csv', index=False
    )
    display(intervention_ablation_summary.style.format(precision=4))


## 7. Final figures


In [ ]:
figure_paths = create_final_figures(
    PROJECT_ROOT,
    artifacts,
    FIGURE_DIR,
    intervention=intervention,
    incorrect_tests=incorrect_tests,
)
for path in figure_paths:
    print(path.name)


## 8. Analysis conclusion and interpretation boundary


In [ ]:
classification_means = classification_table.set_index('model')['accuracy_mean']
faithfulness_means = faithfulness_table.set_index('model')
stability_means = stability_table.set_index('model')
lines = [
    '# Final three-model results',
    '',
    '## Classification',
    *[
        f'- {model}: mean grouped-OOF accuracy {classification_means[model]:.4f}.'
        for model in ('ResNet18', 'DINOv2', 'UNI')
    ],
    '- UNI and ResNet18 had comparable paired per-seed accuracy; both exceeded DINOv2.',
    '',
    '## Faithfulness',
    *[
        (
            f'- {model}: mean attribution-occlusion Spearman '
            f"{faithfulness_means.loc[model, 'attribution_occlusion_spearman_mean']:.4f}; "
            f"top-minus-random logit AUC "
            f"{faithfulness_means.loc[model, 'top_minus_random_target_logit_auc_mean']:.4f}; "
            f"margin AUC "
            f"{faithfulness_means.loc[model, 'top_minus_random_margin_auc_mean']:.4f}."
        )
        for model in ('ResNet18', 'DINOv2', 'UNI')
    ],
    '- UNI had the strongest primary faithfulness results under paired tests, final-family Holm correction, and source-case hierarchical intervals.',
    '- ResNet18 absolute top-deletion drops were large, but its top-minus-random AUCs were positive; random deletion caused greater damage on average.',
    '',
    '## Stability',
    *[
        (
            f'- {model}: prediction agreement '
            f"{stability_means.loc[model, 'pairwise_prediction_agreement_mean']:.4f}; "
            f"common true-class map Spearman "
            f"{stability_means.loc[model, 'mean_common_true_class_spearman_mean']:.4f}."
        )
        for model in ('ResNet18', 'DINOv2', 'UNI')
    ],
    '- DINOv2 had the most stable attribution maps; UNI had the highest mean prediction agreement.',
    '',
    '## Intervention ablation',
    '- Penultimate-layer within-image-mean activation patching was less faithful and less stable than UNI gradient-weighted rollout.',
    '- The causal-ranking loss is not justified and was not implemented.',
    '- The 55-error analysis is exploratory and is reported as a separate statistical family.',
    '- No conclusion is made about other layers, replacement activations, or activation-patching designs.',
    '',
    '## Interpretation',
    '- Highlighted patches are regions contributing to model predictions, not biological causes.',
    '- No pathology-level causal conclusion is supported without external annotations.',
]
summary_path = OUTPUT_DIR / 'final_results_summary.md'
summary_path.write_text('\n'.join(lines) + '\n')
print(summary_path.read_text())
